<a href="https://colab.research.google.com/github/adenikeadewumi/Python-programming-for-ML-WIEOAU/blob/main/09_advanced_python/09_advanced_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 09 — Advanced Python

**Learning Objectives:** Comprehensions, generators, context managers, type hints, useful standard library tools

**Estimated time:** 60–90 minutes

---

## 9.1 Comprehensions

**What is a comprehension?**
A comprehension is a concise, readable way to build a new collection (list, dict, or set) by transforming or filtering an existing one — all in a single line.

**The pattern:**
```
[expression  for  item  in  iterable  if  condition]
 ^what to    ^loop                    ^optional filter
  produce    variable
```

**Why use comprehensions?**
- More readable than a for-loop + append pattern for simple transformations
- Faster — Python executes comprehensions more efficiently than equivalent loops
- The `if` filter lets you select only items that meet a condition
- Works for lists `[]`, dicts `{}`, and sets `{}`

**When NOT to use them:**
If the logic is complex — more than one condition, nested transformations, side effects — use a regular for-loop. Readability always wins over cleverness.

In [ ]:
# The for-loop way vs the comprehension way — same result
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Old way: for-loop + append
squares_loop = []
for n in numbers:
    squares_loop.append(n ** 2)

# Comprehension way: cleaner, faster
squares_comp = [n ** 2 for n in numbers]

print("For-loop:", squares_loop)
print("Comprehension:", squares_comp)

# With a filter — only even numbers
even_squares = [n ** 2 for n in numbers if n % 2 == 0]
print("Even squares:", even_squares)

# Transform strings
names = ["  alice  ", "  BOB  ", "carol  "]
clean = [name.strip().title() for name in names]
print("Cleaned names:", clean)

In [ ]:
# Dict comprehension — build a dictionary in one line
# Pattern: {key_expr: value_expr  for  item  in  iterable  if  condition}

# Word lengths
words = ["python", "machine", "learning", "data", "science"]
lengths = {word: len(word) for word in words}
print("Word lengths:", lengths)

# Reverse a dictionary (swap keys and values)
original = {"a": 1, "b": 2, "c": 3}
reversed_d = {v: k for k, v in original.items()}
print("Reversed:", reversed_d)

# Set comprehension — unique values only
sentence = "the quick brown fox jumps over the lazy brown dog"
unique_words = {word for word in sentence.split()}
print(f"Unique words ({len(unique_words)}):", sorted(unique_words))

In [ ]:
# Nested comprehension — a list of lists
# Useful for creating matrices, grids, combinations

# 3x3 multiplication table as a nested list
table = [[i * j for j in range(1, 4)] for i in range(1, 4)]
for row in table:
    print(row)

print()

# All pairs from two lists (like nested for-loops)
colours = ["red", "blue"]
sizes   = ["S", "M", "L"]
combos  = [(c, s) for c in colours for s in sizes]
print("Combinations:", combos)

# TIP: When comprehensions get complex, use a regular loop for clarity
# This is the same as above but clearer for a beginner reading your code:
combos2 = []
for c in colours:
    for s in sizes:
        combos2.append((c, s))

## 9.2 Generators

**What is a generator?**
A generator is like a list comprehension, but **lazy** — it produces values one at a time, on demand, instead of computing and storing all values upfront.

**Why does this matter?**
Imagine reading a 10-gigabyte log file. A list would load all 10GB into RAM. A generator reads one line, yields it, then moves on — using only a few kilobytes of memory at any moment.

**Two ways to create generators:**
1. **Generator expression:** like a list comprehension but with `()` instead of `[]`
2. **Generator function:** a regular function that uses `yield` instead of `return`

**The `yield` keyword:**
When Python hits `yield`, it pauses the function, hands the value to the caller, and remembers exactly where it left off. The next time the generator is called, it resumes from that exact point.

**In ML/data science:**
Generators are everywhere in deep learning — data loaders yield batches of training data one at a time, so you never need to load an entire dataset into memory at once.

In [ ]:
# Generator expression vs list comprehension
import sys

# List: computes and stores ALL million values immediately
list_result = [x ** 2 for x in range(1_000_000)]

# Generator: stores only the recipe, computes each value when asked
gen_result = (x ** 2 for x in range(1_000_000))

print(f"List size:      {sys.getsizeof(list_result):,} bytes")
print(f"Generator size: {sys.getsizeof(gen_result):,} bytes")
print(f"Memory saving:  {sys.getsizeof(list_result) / sys.getsizeof(gen_result):.0f}x")

print()
# Getting values from a generator
gen = (x ** 2 for x in range(5))
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 4
# Or loop over it
for val in (x**2 for x in range(3)):
    print(val, end=" ")
print()

In [ ]:
# Generator function using yield
# Use this when the logic is too complex for a one-liner expression

def fibonacci():
    """Yields Fibonacci numbers forever — an infinite sequence."""
    a, b = 0, 1
    while True:       # infinite loop is fine — generator pauses at yield
        yield a       # pause here, give 'a' to caller, remember state
        a, b = b, a + b   # resume here next time

# Only compute what we need
fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print("First 10 Fibonacci:", first_10)

print()

# Practical generator — process a large dataset in batches
# This is exactly how deep learning data loaders work
def batch_generator(data, batch_size):
    """Yield successive batches from a list without copying the data."""
    for start in range(0, len(data), batch_size):
        yield data[start : start + batch_size]

dataset = list(range(1, 21))   # simulate 20 data points
print("Processing in batches of 4:")
for batch_num, batch in enumerate(batch_generator(dataset, batch_size=4), 1):
    print(f"  Batch {batch_num}: {batch}")

## 9.3 Context Managers

**What is a context manager?**
A context manager is an object that sets something up before a block of code runs, and tears it down after — even if an error occurs in the middle. The `with` statement is how you use one.

**The classic example — file handling:**
Without `with`:
```python
f = open("file.txt")
data = f.read()    # if this raises an exception, the file is NEVER closed
f.close()          # this line might never run!
```
With `with`:
```python
with open("file.txt") as f:
    data = f.read()   # file closes automatically, even if this crashes
```

**Why context managers matter:**
They guarantee cleanup happens. In data engineering and ML, this is critical for:
- File handles (always close files)
- Database connections (always release the connection)
- GPU memory (always free resources after training)
- Locks in multithreaded code (always release the lock)

**Creating your own context manager:**
Use the `@contextmanager` decorator with a generator function. The code before `yield` is the setup. The code after `yield` (in the finally block) is the teardown.

In [ ]:
# Built-in context managers you already know
# File I/O — file is always closed, even if an exception occurs
with open("/tmp/demo.txt", "w") as f:
    f.write("Hello from context manager!")

with open("/tmp/demo.txt") as f:
    print(f.read())

print()

# Creating your own context manager
from contextlib import contextmanager
import time

@contextmanager
def timer(label=""):
    """Context manager that measures how long a block of code takes."""
    print(f"  [{label}] Starting...")
    start = time.time()
    
    try:
        yield   # everything inside the 'with' block runs here
    finally:
        # This runs after the 'with' block, even if an error occurred
        elapsed = time.time() - start
        print(f"  [{label}] Finished in {elapsed:.4f}s")

with timer("List comprehension"):
    result = [x**2 for x in range(100_000)]

with timer("Manual loop"):
    result2 = []
    for x in range(100_000):
        result2.append(x**2)

## 9.4 Type Hints

**What are type hints?**
Type hints are annotations that tell readers (and tools) what types a function expects and returns. They look like `def greet(name: str) -> str`. Python does NOT enforce them at runtime — they are documentation for humans and tools, not constraints on the interpreter.

**Why use type hints?**
- Your code editor gives you autocompletion and catches type errors before you run
- Other developers (and future you) immediately understand what a function expects
- Tools like `mypy` can statically check your code for type errors
- They are standard practice in professional Python code, especially libraries

**In ML/data science:**
Type hints are especially valuable for data pipeline functions where the distinction between a raw value, a numpy array, a pandas Series, and a list can cause subtle bugs.

In [ ]:
from typing import List, Dict, Optional, Tuple, Union

# Without type hints — unclear what this function expects
def calculate_stats(numbers):
    return sum(numbers) / len(numbers), min(numbers), max(numbers)

# With type hints — immediately clear
def calculate_stats_typed(numbers: List[float]) -> Tuple[float, float, float]:
    """Return (mean, min, max) for a list of numbers."""
    return sum(numbers) / len(numbers), min(numbers), max(numbers)

# Optional — the value can be the type OR None
def find_user(user_id: int) -> Optional[str]:
    """Return username, or None if not found."""
    db = {1: "Alice", 2: "Bob", 3: "Carol"}
    return db.get(user_id)   # returns None if key missing

# Union — the value can be one of several types
def process(value: Union[int, float, str]) -> str:
    return str(value).strip()

# Test them
mean, lo, hi = calculate_stats_typed([1.0, 2.5, 3.7, 4.2, 5.1])
print(f"Mean={mean:.2f}, Min={lo}, Max={hi}")

print(find_user(1))    # "Alice"
print(find_user(99))   # None

## 9.5 Useful Standard Library Tools

**Why the standard library matters:**
Python's standard library ("batteries included") has tools for almost everything. Before installing a third-party package, check if the standard library already has what you need. Three modules you will use constantly in data work:

- `collections` — specialised container types
- `itertools` — efficient tools for looping and combining iterables
- `functools` — higher-order functions (like caching)

In [ ]:
from collections import Counter, defaultdict, namedtuple
import itertools
from functools import lru_cache

# Counter — counts occurrences
words = "the quick brown fox jumps over the lazy dog the fox".split()
freq = Counter(words)
print("Most common:", freq.most_common(3))
print("Count of 'the':", freq["the"])

print()

# defaultdict — dict with a default value for missing keys
# Never need to check 'if key in dict' before appending
from collections import defaultdict

groups = defaultdict(list)
for name, dept in [("Alice","Eng"), ("Bob","HR"), ("Carol","Eng"), ("David","HR")]:
    groups[dept].append(name)
print("Groups:", dict(groups))

print()

# namedtuple — tuple with named fields, like a lightweight class
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 7)
print(f"Point: {p}, x={p.x}, y={p.y}")

print()

# itertools
print("Combinations of 2 from [A,B,C,D]:")
print(list(itertools.combinations("ABCD", 2)))

print("Permutations of 2 from [A,B,C]:")
print(list(itertools.permutations("ABC", 2)))

# lru_cache — automatic memoisation (caching)
@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n <= 1: return n
    return fib(n-1) + fib(n-2)

print("fib(35) =", fib(35))   # fast because results are cached

---

## Key Takeaways

- **Comprehensions** replace loops + append for simple transformations — keep them readable
- **Generators** produce values lazily — essential for large data and memory efficiency
- **Context managers** (`with`) guarantee cleanup even when errors occur
- **Type hints** document your code for tools and teammates — Python doesn't enforce them
- The **standard library** is full of gems — `Counter`, `defaultdict`, `lru_cache`, `itertools`

## Exercises

[09_exercises.ipynb](exercises/09_exercises.ipynb) | [09_solutions.ipynb](exercises/09_solutions.ipynb)

## Next: [10 — NumPy](../10_numpy/10_numpy.ipynb)
